## Setup
This section contains the basic configuration and setup for an expermient. Results will be saved to experiments/dataset/[experiment_name].  

In [ ]:
import os 
args = {
'dataset' : 'swat', # Which dataset to evaluate, string. Default options: 'swat', 'wadi', 'smap', 'msl', 'smd', 'ucr' 
'entity' : 'none', # Which entity within the dataset to evaluate, mixed. Default options: 'none', 'all', or name of a specific entity. 'none' will use to first entity in valid_entities list in preprocess.py 
'down_rate' : 5, # Scale of downsampling to use, integer. Default options: 5 for swat/wadi, 1 other datasets.  
'model' : 'ae', # Model to evaluate, string. Default options: 'ae', 'vae', 'lstmvae', 'omnianomaly', 'usad', 'madgan', 'pca', 'knn'
'hidden_dim' : 'default', # Default option: 'default' to use model parameters specified in paper, or integer
'seq_len' : 12, # Length of the input window, integer. Default options: 12, 48, 128
'num_epochs' : 50, # Number of training epochs, integer. Default: 50
'verbose' : False
}
val_split = 0.1 
batch_size = 64

experiment_name = "/dr" + str(args['down_rate']) + "seq" + str(args['seq_len'])
args['experiment_dir'] = "experiments/" + args['dataset'] + experiment_name
print("Experiments will be saved to " + args['experiment_dir'])

os.makedirs(args['experiment_dir'], exist_ok=True) 
if os.path.isfile(args['experiment_dir'] + "/metrics.csv"):
    print("Previously saved metrics detected") 

In [ ]:
from models.AE import ae_experiment
from models.VAE import vae_experiment
from models.LSTMVAE import lstmvae_experiment
from models.Omnianomaly import omni_experiment
from models.USAD import usad_experiment
from models.MADGAN import madgan_experiment
from models.PCA import pca_experiment 
from models.KNN import knn_experiment 

#Stores and keeps track of possible experiments 
model_zoo = {
    'ae': ae_experiment, 
    'vae': vae_experiment, 
    'lstmvae': lstmvae_experiment, 
    'omnianomaly': omni_experiment, 
    'usad' : usad_experiment, 
    'madgan' : madgan_experiment, 
    'pca': pca_experiment, 
    'knn' : knn_experiment,
}

## Single entity 
This section evaluates a single entity. Results are saved to metrics.csv within the experiment folder.

In [ ]:
from utils.preprocess import get_data, valid_entities
from utils.utils import seq_data, save_metrics
from torch.utils.data import DataLoader
assert args['entity'] != 'all', "This block evaluates a single entitiy." 

experiment = model_zoo[args['model']]

entity = args['entity']
if entity == 'none':
    entity = None 

#Load data
train_data, val_data, test_data, test_labels = get_data(args['dataset'], val_split = val_split, 
                                                        seq_len = args['seq_len'], down_rate = args['down_rate'], 
                                                        entity = entity, verbose = args['verbose'])
    
input_dim = train_data.shape[-1]
args['input_dim'] = input_dim
    
train_dataset = seq_data(train_data, args['seq_len'])
val_dataset = seq_data(val_data, args['seq_len'])
test_dataset = seq_data(test_data, args['seq_len'], test_labels) 

train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = batch_size, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = batch_size)

#Evaluate 
if args['model'] in ['pca', 'knn']:
    metrics = experiment(train_data, test_data, test_labels, args)
else:    
    metrics = experiment(train_loader, val_loader, test_loader, args)
    
metrics['entity'] = entity 
save_metrics(metrics, args)

## Multi-entity 
This section evaluates all entities in a dataset. Results are saved to metrics.csv within the experiment folder. ucr_skip records the entities that were skipped due to computational restraints in the original paper. 

In [ ]:
from utils.preprocess import get_data, valid_entities, ucr_skip
from utils.utils import seq_data, save_metrics
from torch.utils.data import DataLoader
assert args['entity'] == 'all', "This block evaluates all entities." 

experiment = model_zoo[args['model']]


if args['dataset'] in valid_entities.keys():
    entity = valid_entities[args['dataset']]
else:
    entity = None #Swat/Wadi do not have multiple entities   

for e in entity: 
    if args['dataset'] == "ucr" and e in ucr_skip:
        print("Dataset too large")
        continue
        
    #Load data
    train_data, val_data, test_data, test_labels = get_data(args['dataset'], val_split = val_split, 
                                                            seq_len = args['seq_len'], down_rate = args['down_rate'], 
                                                            entity = e, verbose = args['verbose'])

    input_dim = train_data.shape[-1]
    args['input_dim'] = input_dim

    train_dataset = seq_data(train_data, args['seq_len'])
    val_dataset = seq_data(val_data, args['seq_len'])
    test_dataset = seq_data(test_data, args['seq_len'], test_labels) 

    train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
    val_loader = DataLoader(val_dataset, batch_size = batch_size, shuffle = True)
    test_loader = DataLoader(test_dataset, batch_size = batch_size)

    #Evaluate 
    if args['model'] in ['pca', 'knn']:
        metrics = experiment(train_data, test_data, test_labels, args)
    else:    
        metrics = experiment(train_loader, val_loader, test_loader, args)

    metrics['entity'] = e
    save_metrics(metrics, args)

## Metrics and statistics 
This section reads metrics.csv to calculate the mean and std of the results, and performs the Friedman test. 

In [ ]:
import pandas as pd 
# Viewing Results and Statistics 
results_df = pd.read_csv(experiment_dir + "/metrics.csv")
models = results_df["model"].unique()
entities = results_df["entity"].unique()

print("Counts")
print(results_df.groupby("model", as_index=False).count())

print("Stats")
print(results_df.groupby("model", as_index=False).agg(mean_score=('score', 'mean'),
                                                       std_score=('score', 'std')))

In [ ]:
from autorank import autorank, plot_stats, create_report, latex_table
import matplotlib.pyplot as plt

# Friedman Test
ranking_dict = {}
for m in models: 
    score_list = []
    for e in entities: 
        score_list.append(results_df[(results_df["model"] == m) & (results_df["entity"] == e)]["score"].values[0])
    ranking_dict[m] = score_list

ranking_df = pd.DataFrame(ranking_dict)
rank_result = autorank(ranking_df, alpha=0.05, verbose=False, force_mode="nonparametric")

plot_stats(rank_result, allow_insignificant = True)
plt.show()

## UCR Groups 
This segment performs the Friedman test on groups within the UCR dataset. Groups are formed by splitting UCR entities by task as discussed in Appendix A. 

In [ ]:
from autorank import autorank, plot_stats, create_report, latex_table
import matplotlib.pyplot as plt
from utils.preprocess import ucr_groups
# UCR groups 
group = 1

entities = ucr_groups[group]
ranking_dict = {}
for m in models: 
    score_list = []
    for e in entities: 
        score = results_df[(results_df["model"] == m) & (results_df["entity"] == (e))]["score"].values
        if len(score) != 1:
            continue
        else:
            score_list.append(score[0])
    ranking_dict[m] = score_list
    
ranking_df = pd.DataFrame(ranking_dict)
rank_result = autorank(ranking_df, alpha=0.05, verbose=False, force_mode="nonparametric")

plot_stats(rank_result, allow_insignificant = False)
plt.show()